# 93 - Introducción a Gradio — Base

Gradio permite crear **interfaces web interactivas** para modelos de IA.

Este ejemplo usa un clasificador Iris con scikit-learn.

In [ ]:
try:
    import gradio as gr
    from sklearn.datasets import load_iris
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    import numpy as np

    HAS_GRADIO = True
    print("✅ Gradio importado")
except Exception as e:
    HAS_GRADIO = False
    print("⚠️ Gradio o scikit-learn no disponible:", e)


Esta celda entrena el clasificador Iris, define la función de predicción y construye la interfaz básica de Gradio. Si Gradio o scikit-learn no están disponibles, se omite la demo para que el notebook no falle.


In [ ]:
if HAS_GRADIO:
    data = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        data.data, data.target, random_state=0
    )
    model = LogisticRegression(max_iter=200).fit(X_train, y_train)

    def predict(sepal_length, sepal_width, petal_length, petal_width):
        X = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
        return data.target_names[model.predict(X)[0]]

    demo = gr.Interface(
        fn=predict,
        inputs=[
            gr.Number(label="Sepal length"),
            gr.Number(label="Sepal width"),
            gr.Number(label="Petal length"),
            gr.Number(label="Petal width"),
        ],
        outputs=gr.Textbox(label="Predicción"),
    )
    print("Ejecuta demo.launch() para abrir la interfaz.")


### Ejercicio
1. Lanza la demo con `demo.launch()`.
2. Sustituye el modelo por un árbol de decisión.
3. Personaliza la interfaz con otros widgets.

## Ampliación progresiva

Los ejemplos siguientes hacen crecer la demo desde una predicción simple hasta una interfaz más útil:

1. Comparar varios modelos.
2. Mostrar probabilidades, no solo la etiqueta final.
3. Crear una interfaz con controles más adecuados.
4. Añadir ejemplos de entrada para probar la aplicación rápido.

Esta celda entrena el clasificador Iris, define la función de predicción y construye la interfaz básica de Gradio. Si Gradio o scikit-learn no están disponibles, se omite la demo para que el notebook no falle.


In [ ]:
if HAS_GRADIO:
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.neighbors import KNeighborsClassifier

    models = {
        "Regresión logística": LogisticRegression(max_iter=200).fit(X_train, y_train),
        "Árbol de decisión": DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train),
        "K vecinos": KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train),
    }

    def predict_with_model(model_name, sepal_length, sepal_width, petal_length, petal_width):
        selected = models[model_name]
        X = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
        pred = selected.predict(X)[0]
        return data.target_names[pred]

    print(predict_with_model("Árbol de decisión", 5.1, 3.5, 1.4, 0.2))

Esta celda entrena el clasificador Iris, define la función de predicción y construye la interfaz básica de Gradio. Si Gradio o scikit-learn no están disponibles, se omite la demo para que el notebook no falle.


In [ ]:
if HAS_GRADIO:
    def predict_probabilities(model_name, sepal_length, sepal_width, petal_length, petal_width):
        selected = models[model_name]
        X = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
        if hasattr(selected, "predict_proba"):
            probs = selected.predict_proba(X)[0]
        else:
            probs = np.zeros(len(data.target_names))
            probs[selected.predict(X)[0]] = 1.0
        return {name: float(prob) for name, prob in zip(data.target_names, probs)}

    iris_examples = [
        ["Regresión logística", 5.1, 3.5, 1.4, 0.2],
        ["Árbol de decisión", 6.0, 2.2, 4.0, 1.0],
        ["K vecinos", 6.9, 3.1, 5.4, 2.1],
    ]

    demo_avanzada = gr.Interface(
        fn=predict_probabilities,
        inputs=[
            gr.Dropdown(list(models.keys()), value="Regresión logística", label="Modelo"),
            gr.Slider(4.0, 8.0, value=5.1, step=0.1, label="Sepal length"),
            gr.Slider(2.0, 4.5, value=3.5, step=0.1, label="Sepal width"),
            gr.Slider(1.0, 7.0, value=1.4, step=0.1, label="Petal length"),
            gr.Slider(0.1, 2.5, value=0.2, step=0.1, label="Petal width"),
        ],
        outputs=gr.Label(num_top_classes=3, label="Probabilidades"),
        examples=iris_examples,
        title="Comparador de modelos Iris",
        description="Prueba varios clasificadores y compara su confianza.",
    )
    print("Ejecuta demo_avanzada.launch() para abrir la versión ampliada.")

### Reto adicional

Añade una función que reciba varias flores a la vez y devuelva una tabla con la predicción de cada una. Pista: usa `gr.Dataframe`.

## Para profundizar

Este notebook muestra lo básico de Gradio. La **documentación completa** (`Gradio_Documentacion.md`) incluye:

- **Estado y variables globales** (`gr.Variable`) para mantener estado entre llamadas.
- **Eventos encadenados** (`change`, `click`, `submit`) para crear flujos interactivos.
- **Caching** (`cache_examples=True`) para no reprocesar ejemplos conocidos.
- **Streaming** de respuestas para generar texto token a token.
- **Blocks API** (`gr.Blocks`) para interfaces personalizadas con filas, columnas, pestañas.
- **Flagging** para que usuarios puedan marcar respuestas incorrectas.
- **Themes** y personalización visual.
- **Despliegue** en Hugging Face Spaces, Docker, o con API REST.

Consulta el documento en la carpeta `Documentacion/` para ver ejemplos de这些 funcionalidades avanzadas.

In [ ]:
# Ejemplo avanzado: Blocks con estado (no ejecutable sin Gradio completo)
# Esto muestra el patrón, pero requiere Gradio instalado para funcionar

# contar_estado = gr.State(value=0)
# with gr.Blocks() as demo_con_estado:
#     btn = gr.Button("Incrementar")
#     output = gr.Number()
#     btn.click(lambda n: n + 1, inputs=contar_estado, outputs=output)
#     output.change(lambda n: n, inputs=output, outputs=contar_estado)

# Para ver más: consulta Gradio_Documentacion.md sección 'Cosas que ofrece Gradio y no se han mentionado'